In [2]:
import xarray as xr

# Load the dataset
ds = xr.open_dataset('/home/csutter/NYSM/netcdf/proc/2024/01/20240101.nc')

# See what's inside (Dimensions, Coordinates, and Variables)
print(ds)

# Option A: Just the names
print(list(ds.data_vars))

<xarray.Dataset>
Dimensions:               (station: 126, time_5M: 288)
Coordinates:
  * station               (station) object 'ADDI' 'ANDE' ... 'WOLC' 'YORK'
  * time_5M               (time_5M) datetime64[ns] 2024-01-01 ... 2024-01-01T...
Data variables: (12/37)
    lat                   (station) float32 ...
    lon                   (station) float32 ...
    elev                  (station) float32 ...
    tair                  (station, time_5M) float32 ...
    ta9m                  (station, time_5M) float32 ...
    tslo                  (station, time_5M) float32 ...
    ...                    ...
    sm25                  (station, time_5M) float32 ...
    sm50                  (station, time_5M) float32 ...
    frozen05              (station, time_5M) float64 ...
    frozen25              (station, time_5M) float64 ...
    frozen50              (station, time_5M) float64 ...
    snow_depth            (station, time_5M) float32 ...
Attributes:
    Conventions:     CF-1.6
    fea

In [32]:
import xarray as xr
import os
from glob import glob

# 1. Setup your file list and a place to store results
# Use 2024 and 2025 b/c didn't have labeled data from those dates
dir2024 = glob("/home/csutter/NYSM/netcdf/proc/2024/*/*")
print(len(dir2024)) # leap year, 366 days
dir2025 = glob("/home/csutter/NYSM/netcdf/proc/2025/*/*")
print(len(dir2025))
files = dir2024+dir2025
print(len(files))

# dry_days_found = []

results = []

# 2. Loop through every file (each file = one day)
for filename in files:
    # print(filename)
    ds = xr.open_dataset(filename)
    
    # Let's assume your dimensions are named 'station' and 'time'
    # We grab the first index (0) and the last index (-1) of the time dimension
    # Note that this is the first and last time in the ds, which is the same for every station, so we're just grabbing that time "slice" that contains all stations. Efficiency of working with netcdf
    first_entry = ds.isel(time_5M=0)
    last_entry = ds.isel(time_5M=-1)
    
    # Calculate the change over the day for ALL stations at once
    # Note that this operation is happening as a list operation, where each element is a station. The lengths of snow_change will be 126, for example.
    snow_change = last_entry['snow_depth'] - first_entry['snow_depth']
    precip_change = last_entry['precip'] - first_entry['precip']
    
    # # Find stations where BOTH changes are exactly 0
    # # This creates a list of True/False values for each station
    # is_dry = (snow_change <= 0) & (precip_change <= 0)
    
    # # Identify which stations were "True"
    # # .where(is_dry, drop=True) removes all the 'False' stations
    # dry_stations = is_dry.where(is_dry == True, drop=True)
    
    # # 3. If we found any, add them to our list
    # for station_id in dry_stations.station.values:
    #     # We grab the date from the 'time' coordinate of the file
    #     date_str = str(ds.time_5M.values[0])[:10] 
    #     dry_days_found.append({'date': date_str, 'station': station_id})
    

    # Get the station names list
    station_names = ds.station.values
    date_str = str(ds.time_5M.values[0])[:10]
    
    # 2. Loop using a range counter (0, 1, 2...) 
    # This is often easier to follow than zipping
    num_stations = len(station_names)
    
    for i in range(num_stations):
        # Extract values for the current station (i)
        current_station = station_names[i]
        s_delta = float(snow_change[i])
        p_delta = float(precip_change[i])
        
        # 3. Logic check: If it was a "dry" day
        # if s_delta <= 0 and p_delta <= 0:
        entry = {
            'date': date_str,
            'station': current_station,
            'snow_delta': s_delta,
            'precip_delta': p_delta
        }
        results.append(entry)
    ds.close() # Good practice to close the file before the next loop


print(results[0:5])
# # Convert to a table for easy viewing
# df = pd.DataFrame(results)
# print(df.head())

366
365
731
[{'date': '2024-10-14', 'station': 'ADDI', 'snow_delta': -0.0025216154754161835, 'precip_delta': -22.160001754760742}, {'date': '2024-10-14', 'station': 'ANDE', 'snow_delta': -0.0007899074698798358, 'precip_delta': 2.0500049591064453}, {'date': '2024-10-14', 'station': 'BATA', 'snow_delta': -0.002336853416636586, 'precip_delta': 9.099994659423828}, {'date': '2024-10-14', 'station': 'BEAC', 'snow_delta': nan, 'precip_delta': -1.160003662109375}, {'date': '2024-10-14', 'station': 'BELD', 'snow_delta': -0.0008053808705881238, 'precip_delta': -9.369998931884766}]


92106

In [69]:
print(len(results))
print(731*126) # check length approx to see that we have grabbed snow and precip depth accum for every station, every date

92701
92106


In [85]:
import pandas as pd
import numpy as np

# Convert to a df for easy viewing
# See NAN investigation below, just ignore those for finding dry dates
df = pd.DataFrame(results)



date             object
station          object
snow_delta      float64
precip_delta    float64
snow_abs        float64
precip_abs      float64
dtype: object


In [ ]:
# investigate nans
# Looks like sometimes the snow depth gauge isn't working. Seemed to be the case for like all of October 2024 for BEAC

d_ex = xr.open_dataset('/home/csutter/NYSM/netcdf/proc/2024/10/20241014.nc')
beac = d_ex.sel(station = 'BEAC')
beac_first = beac.isel(time_5M=287)
beac_last = beac.isel(time_5M=-1)

beac_snow_first = beac_first['snow_depth'].item()
beac_snow_last = beac_last['snow_depth'].item()

print(beac_snow_first)
print(beac_snow_last)


In [100]:
# Prepare dataframe more


### Add absolute value
df["snow_abs"] = np.abs(df["snow_delta"])
df["precip_abs"] = np.abs(df["precip_delta"])

df.head(5)
print(df.dtypes)

# Make a datetime col so we can later filter for winter months, Dec - Mar 
df['datetime'] = pd.to_datetime(df['date'])

print(len(df))


date                    object
station                 object
snow_delta             float64
precip_delta           float64
snow_abs               float64
precip_abs             float64
datetime        datetime64[ns]
dtype: object
92701


In [98]:
# Prepare winter months and top 20 dates

### Make a datetime col so we can filter for winter months, Dec - Mar 

#  Define your "Winter" months
# Dec (12), Jan (1), Feb (2), Mar (3)
winter_months = [12, 1, 2, 3]

#  Filter the DataFrame
# .dt.month extracts the month number from the datetime object
winter_df = df[df['datetime'].dt.month.isin(winter_months)]
winter_df.head(5)


### Grab top 20 dates from each station

lowsnow_dates = []
lowsnow_values = []
lowprecip_dates = []
lowprecip_values = []

stations_run = [] # just for tracking

for st in np.unique(winter_df["station"]):
    dfsub = winter_df[winter_df["station"]==st]
    sort_snow = dfsub.sort_values("snow_abs", ascending = True).reset_index()
    sort_precip = dfsub.sort_values("precip_abs", ascending = True).reset_index()
    for i in range(0,20):
        snowval = sort_snow["snow_abs"].iloc[i]
        snowusedate = sort_snow["date"].iloc[i]
        lowsnow_values.append(snowval)
        lowsnow_dates.append(snowusedate)

        precipval = sort_precip["precip_abs"].iloc[i]
        precipusedate = sort_precip["date"].iloc[i]
        lowprecip_values.append(precipval)
        lowprecip_dates.append(precipusedate)

        stations_run.append(st)


In [99]:
print(len(lowsnow_dates))
print(len(lowprecip_dates))
print(len(stations_run))

print(126*20) # i think maybe there are a few extra stations like profiler stations or something, don't worry about it, just an approx estimate to make sure code worked right

2540
2540
2540
2520


In [96]:
# Convert to DF

# Note, it's a little odd to have the lowsnow and lowprecip in one row, since their dates aren't tied, but the logic here is that they are the top 20 for each. So the first row of ADDI is the corresponding dates for 1) lowest precip and 2) lowest snow
df_nonevents1 = pd.DataFrame({"station":stations_run, "lowsnow_dates":lowsnow_dates, "lowsnow_values":lowsnow_values, "lowprecip_dates":lowprecip_dates, "lowprecip_values":lowprecip_values})

display(df_nonevents1.head(4))
print(len(df_nonevents1))

,station,lowsnow_dates,lowsnow_values,lowprecip_dates,lowprecip_values
0,ADDI,2025-03-07,0.000030,2024-12-26,0.0
1,ADDI,2025-12-16,0.000069,2024-02-12,0.0
2,ADDI,2025-01-30,0.000096,2025-03-11,0.0
3,ADDI,2024-03-28,0.000124,2025-12-06,0.0


2540


In [101]:
# SAVE OUT DATAFRAMES

# Save the dataframe of selected top 20 dry dates - for tracking for when we'll inevitably need to add in cam colocation and regional subsetting based on the non event dates that were valid for that region

df_nonevents1.to_csv("/home/csutter/DRIVE-clean/weather_events/data/nysm_dates/nonevents_top20.csv")

# Also save the general df that tracks accum precip and accum snow for each day for each site -- this is usual to have for future analyses!

print(len(df))
df.to_csv("/home/csutter/DRIVE-clean/weather_events/data/nysm_dates/station_date_accumsnow_accumprecip.csv")

92701


#### Prepare data for operational run
- Source code: /home/csutter/DRIVE-clean/weather_events/notebooks/ncei_dataset_analysis.ipynb

In [128]:

# Grab only unique dates

datesselected = list(df_nonevents1["lowsnow_dates"]) + list(df_nonevents1["lowprecip_dates"])

print(len(df_nonevents1))
print(len(datesselected))

dates_list = np.unique(datesselected)

print(len(dates_list))

2540
5080
240


In [129]:
dates_list[0:4]

array(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04'],
      dtype='<U10')

In [130]:
240*12

2880

In [131]:
# add minutes to the dates

datesrun = []
for d in dates_list:
    for hh in ["0000","0200","0400","0600","0800","1000","1200","1400","1600","1800","2000","2200"]:
        datesrun.append(f"{d}_{hh}")

print(len(datesrun))

print(datesrun[0:4])

# remove the "-" from each date

datesrun = [x.replace("-", "") for x in datesrun]
print(datesrun[0:4])

print(len(datesrun))
print(len(np.unique(datesrun)))

# grab unique only 




2880
['2024-01-01_0000', '2024-01-01_0200', '2024-01-01_0400', '2024-01-01_0600']
['20240101_0000', '20240101_0200', '20240101_0400', '20240101_0600']
2880
2880


In [132]:
print("Unique datetimes from event of int")
print(len(datesrun))

Unique datetimes from event of int
2880


In [ ]:
# Remove dates that have already been ran


In [133]:
ld = glob("/home/csutter/DRIVE-clean/operational_runs/*")
ld = sorted(ld)

inf_sets_ran = ld

dirs_w_preds = [f"{i}/data_6_ensembling/*/*/*/*/*" for i in inf_sets_ran]

# print(dirs_w_preds)

pred_datetimes = []
pred_path = []
for dr in dirs_w_preds:
    # print(dr) # just for checking counts per dir
    # dr_files = [] # just for checking counts per dir
    listpredfiles = glob(dr)
    for fl in listpredfiles:
        # print(fl)
        i = fl.rfind("/")
        filedate = fl[i-13:i]
        pred_datetimes.append(filedate)
        # dr_files.append(filedate) # just for checking counts per dir

        ### Must also grab the tracker path which contains the predictions and the image path (should have all the data in it we need)
        pred_path.append(fl)
    # print(len(dr_files)) # just for checking counts per dir


print(len(pred_datetimes))
print(len(pred_path))


print(pred_datetimes[0:4])

16155
16155
['20250210_1000', '20250204_1000', '20250228_1000', '20250227_1000']


In [138]:
# Cross check from dates_list and remove any dates already ran/running

dates_list_new = []
for d in datesrun:
    if d not in pred_datetimes:
        dates_list_new.append(d)

print("Unique datetimes from event of int")
print(len(datesrun))

print("Unique datetimes removing what's been ran")
print(len(dates_list_new))

Unique datetimes from event of int
2880
Unique datetimes removing what's been ran
1330


In [141]:
dates_list_new[0:3]

# Save out 
forcsv = pd.DataFrame(dates_list_new, columns=['date'])[860:] # ADJUST HERE!! Manually adjust how many instance to run in one sbatch run. Usually ~400 is good (takes 20 hours)
forcsv

savetodir = "/home/csutter/DRIVE-clean/operational_runs/set49_nonevents_nysm3" # ADJUST HERE!! Naming to match what event you're running
os.makedirs(savetodir, exist_ok=True)
forcsv.to_csv(f"{savetodir}/dates.csv") 